In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import pickle
from sklearn.preprocessing import LabelEncoder
import gc # For garbage collection

# --- Step 1: Load Test Data and Saved Artifacts ---
print("Loading test data and saved artifacts...")

# Load the test dataset
test = pd.read_parquet("test_data.parquet")

# Load the trained model (now saved as a pickle file)
try:
    with open("lgbm_model.pkl", "rb") as f:
        model = pickle.load(f)
    print("✅ Trained model 'lgbm_model.pkl' loaded successfully.")
except FileNotFoundError:
    print("ERROR: 'lgbm_model.pkl' not found. Please run the training script first to train and save the model.")
    exit() # Exit if model is not found

# Load the saved feature columns list
try:
    with open("feature_cols.pkl", "rb") as f:
        feature_cols = pickle.load(f)
    print("✅ Feature columns 'feature_cols.pkl' loaded successfully.")
except FileNotFoundError:
    print("ERROR: 'feature_cols.pkl' not found. Please run the training script first to save the feature list.")
    exit()

# Load pre-calculated offer features from training
try:
    # Read as a DataFrame, then set_index if id3 is not already the index
    offer_agg_train = pd.read_parquet('offer_agg.parquet')
    if 'id3' in offer_agg_train.columns:
        offer_agg_train.set_index('id3', inplace=True)
    print("✅ Offer aggregations 'offer_agg.parquet' loaded successfully.")
except FileNotFoundError:
    print("ERROR: 'offer_agg.parquet' not found. Please ensure it's saved by the training script.")
    print("Creating a dummy offer_agg_train for graceful error handling (NOT FOR PRODUCTION USE).")
    # Create an empty DataFrame with the expected columns and index to prevent KeyError
    offer_agg_train = pd.DataFrame(columns=['offer_ctr_smoothed', 'offer_count'])
    offer_agg_train.index.name = 'id3' # Ensure index name is correct

# Load the saved label encoders for f-series
try:
    with open("f_series_label_encoders.pkl", "rb") as f:
        f_series_encoders = pickle.load(f)
    print("✅ F-series label encoders 'f_series_label_encoders.pkl' loaded successfully.")
except FileNotFoundError:
    print("ERROR: 'f_series_label_encoders.pkl' not found. Please run the training script first to save encoders.")
    exit()

# Load the saved label encoder for top spend
try:
    with open("top_spend_encoder.pkl", "rb") as f:
        top_spend_encoder = pickle.load(f)
    print("✅ Top spend encoder 'top_spend_encoder.pkl' loaded successfully.")
except FileNotFoundError:
    print("ERROR: 'top_spend_encoder.pkl' not found. Please run the training script first to save encoder.")
    exit()

# Load the global max time from the training set to prevent data leakage in recency features
try:
    with open("global_max_time.pkl", "rb") as f:
        global_max_time = pickle.load(f)
    print("✅ Global max time 'global_max_time.pkl' loaded successfully.")
except FileNotFoundError:
    print("ERROR: 'global_max_time.pkl' not found. This should be saved from train['id4'].max() in training.")
    # Critical fallback: If not found, use max date in test. This can introduce data leakage
    # or inconsistency if test data extends beyond train data.
    print("Falling back to test data's max date for global_max_time. This might lead to inconsistency in recency features.")
    test['id4'] = pd.to_datetime(test['id4'], errors='coerce')
    global_max_time = test['id4'].max()


print("✅ All artifacts loaded successfully.")


# --- Step 2: Recreate Features on Test Set (Applying Training Logic) ---
print("Starting feature engineering on the test set...")

# Store original IDs for submission
original_ids = test[['id1', 'id2', 'id3', 'id5']].copy()

# Consistent data types for IDs
test['id2'] = test['id2'].astype(str)
test['id3'] = test['id3'].astype(str)

# --- Offer-level features ---
# Merge with pre-calculated offer features. reset_index() is needed if id3 was the index when saved.
# Ensure 'id3' is a column in both test and offer_agg_train for merge.
test = test.merge(offer_agg_train.reset_index(), on='id3', how='left')
# Fill NaNs for new offers not seen during training
global_ctr_fallback = 0.01 # This should ideally be the global_ctr from training
test['offer_ctr_smoothed'] = test['offer_ctr_smoothed'].fillna(global_ctr_fallback)
test['offer_count'] = test['offer_count'].fillna(0) # This line should now work after fixing offer_agg loading/saving


# --- Time-based features ---
test['id5'] = pd.to_datetime(test['id5'], errors='coerce')
test['id4'] = pd.to_datetime(test['id4'], errors='coerce')
test['day_of_week'] = test['id5'].dt.dayofweek.fillna(-1).astype(int)
test['hour'] = test['id4'].dt.hour.fillna(-1).astype(int)


# --- User-Offer interaction features ---
test['user_offer_seen_count'] = test.groupby(['id2', 'id3'])['id3'].transform('size')
test['user_offer_rank'] = test.groupby('id2')['user_offer_seen_count'].rank(method='dense', ascending=False)
test['seen_bucket'] = pd.cut(
    test['user_offer_seen_count'],
    bins=[-1, 1, 3, 5, np.inf],
    labels=[0, 1, 2, 3],
    right=True # Ensure consistency with training
).astype(int)


# --- User-level and Recency features ---
test['last_seen'] = test.groupby(['id2', 'id3'])['id4'].transform('max')
# Use the global_max_time loaded from training for consistent recency calculation
test['days_since_seen'] = (global_max_time - test['last_seen']).dt.days.fillna(-1).astype(int)
test['recency_rank'] = test.groupby('id2')['last_seen'].rank(ascending=False)

user_agg_test = test.groupby('id2').agg(
    user_offer_count=('id3', 'nunique'),
    last_user_event=('id4', 'max')
)
test = test.merge(user_agg_test, on='id2', how='left')

test['user_recency'] = (global_max_time - test['last_user_event']).dt.total_seconds() / (60*60*24)
# Add a small epsilon to denominator to prevent division by zero
test['user_offer_seen_freq'] = test['user_offer_seen_count'] / (test['user_offer_count'] + 1e-6)

# Drop intermediate columns if they are not part of feature_cols
test.drop(columns=['last_seen', 'last_user_event'], inplace=True, errors='ignore')


# --- f-series features ---
# These lists are determined by your training script's feature engineering
# and the saved label encoders.
categorical_f_cols_from_encoder = list(f_series_encoders.keys())
f_cols_range = [f"f{i}" for i in range(1, 367)] # Represents all possible f-cols

# Process categorical f-series using saved encoders
for col in categorical_f_cols_from_encoder:
    if col in test.columns:
        test[col] = test[col].fillna("missing").astype(str)
        known_categories = f_series_encoders[col].classes_
        # IMPORTANT: Now, 'missing' should always be in 'known_categories' due to training script fix.
        # Apply transformation robustly: map unseen to 'missing' if 'missing' was a training category
        test[col] = test[col].apply(lambda x: x if x in known_categories else 'missing')
        test[col] = f_series_encoders[col].transform(test[col])
    else:
        # If a categorical feature is completely missing in test, fill with the encoded 'missing' value
        print(f"WARNING: Categorical feature '{col}' not found in test data. Adding with encoded 'missing' value.")
        # This assumes 'missing' is a known class, which it should be after the training script fix.
        test[col] = f_series_encoders[col].transform(['missing'])[0]

# Process numerical f-series
# Iterate through all 'f' columns that are expected to be numerical
numerical_f_cols = list(set(f_cols_range) - set(categorical_f_cols_from_encoder))
for col in numerical_f_cols:
    if col in test.columns:
        test[col] = pd.to_numeric(test[col], errors='coerce').fillna(-1).astype('float32')
    else:
        # If a numerical feature is completely missing in test, add it with a default value
        print(f"WARNING: Numerical feature '{col}' not found in test data. Adding with default -1.")
        test[col] = -1.0


# --- Spend-based features ---
category_30 = [f'f{i}' for i in range(152, 163)]
category_180 = [f'f{i}' for i in range(163, 174)]

# Ensure these columns exist before summing (if not, sum will yield NaN)
# For robustness, fill missing f cols that are part of category_30/category_180 before summing
for c_list in [category_30, category_180]:
    for col in c_list:
        if col not in test.columns:
            # If a category column is entirely missing in test data, assume 0 spend for it.
            test[col] = 0.0
        else:
            # For existing columns, convert to numeric and fill any internal NaNs with 0.0.
            test[col] = pd.to_numeric(test[col], errors='coerce').fillna(0.0)


test['total_spend_30'] = test[category_30].sum(axis=1)
test['total_spend_180'] = test[category_180].sum(axis=1)

# Handle potential division by zero by adding a small epsilon
test[[col + '_ratio_30' for col in category_30]] = test[category_30].div(test['total_spend_30'] + 1e-6, axis=0)
test[[col + '_ratio_180' for col in category_180]] = test[category_180].div(test['total_spend_180'] + 1e-6, axis=0)

# idxmax can return NaN if all values in the row are NaN or if they are all equal.
# Fill with 'missing_category' from training, then convert to string.
test['top_spend_30'] = test[category_30].idxmax(axis=1).fillna("missing_category").astype(str)
test['top_spend_180'] = test[category_180].idxmax(axis=1).fillna("missing_category").astype(str)

short_term_spend = test[category_30]
long_term_spend = test[category_180]
long_term_spend.columns = short_term_spend.columns
test[[f'shift_{col}' for col in category_30]] = short_term_spend / (long_term_spend + 1e-6)


# Apply saved top_spend_encoder
known_spend_cats = top_spend_encoder.classes_
# Map unseen top_spend categories to 'missing_category' (this string must be in known_spend_cats)
# IMPORTANT: 'missing_category' should now always be in known_spend_cats due to training script fix.
test['top_spend_30_encoded'] = test['top_spend_30'].apply(lambda x: x if x in known_spend_cats else 'missing_category')
test['top_spend_180_encoded'] = test['top_spend_180'].apply(lambda x: x if x in known_spend_cats else 'missing_category')

test['top_spend_30_encoded'] = top_spend_encoder.transform(test['top_spend_30_encoded'])
test['top_spend_180_encoded'] = top_spend_encoder.transform(test['top_spend_180_encoded'])

print("✅ Feature engineering complete.")


# --- Step 3: Predict and Save Submission ---
print("Generating predictions...")

# Prepare the test DataFrame for prediction with the exact features the model expects
X_test = test[feature_cols].copy() # .copy() to avoid SettingWithCopyWarning later

# Ensure no NaNs are passed to the model. Fill any remaining with -1 (or appropriate default).
# This is crucial as LightGBM's handling of NaNs might differ from what's explicitly filled.
# It's safer to ensure the data is clean before prediction.
X_test = X_test.fillna(-1)

# Predict probabilities using the loaded LGBMClassifier model
# predict_proba returns an array [prob_class_0, prob_class_1]. We want prob_class_1.
original_ids['pred'] = model.predict_proba(X_test)[:, 1]

# Format for submission
submission = original_ids[['id1', 'id2', 'id3', 'id5', 'pred']]
submission.to_csv("submission.csv", index=False)

print(f"✅ Final submission created: submission.csv with shape {submission.shape}")

# Clean up memory
del test, X_test, submission, original_ids
del offer_agg_train, f_series_encoders, top_spend_encoder, model
gc.collect()

NameError: name 'f_series_label_encoders' is not defined